# PCA Workflow With and Without PCA

This notebook shows a simple, direct machine learning flow using plain code cells.

## What this notebook does

1. Create a dummy classification dataset with 10 features.
2. Add a few null values so the cleaning step is visible.
3. Remove null values and standardize the features.
4. Train a baseline logistic regression model and print accuracy.
5. Apply PCA with `n_components=2`.
6. Train the same model on the PCA output and print accuracy.

Why this layout?
- each cell has one job
- no custom functions are used
- markdown explains the purpose of each step

In [16]:
import numpy as np
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

## 1. Create a dummy dataset

We start with a synthetic dataset so the PCA workflow is easy to understand.

Dataset design:
- 10 sample numeric features
- binary target column
- a few null values added on purpose for preprocessing practice

In [17]:
# Create a binary classification dataset with 10 features.
X, y = make_classification(
    n_samples=300,
    n_features=10,
    n_informative=6,
    n_redundant=2,
    n_classes=2,
    random_state=42,
    shuffle=True,
    )

feature_cols = [f"feature_{i}" for i in range(1, 11)]
df = pd.DataFrame(X, columns=feature_cols)
df["target"] = y

# Add a few null values so the cleaning step is meaningful.
df.loc[[3, 18, 41], "feature_2"] = np.nan
df.loc[[7, 25], "feature_7"] = np.nan

print("Dataset shape before cleaning:", df.shape)
df.head()

Dataset shape before cleaning: (300, 11)


,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,target
0,-8.263516,2.488870,-0.951184,2.880649,-3.238791,-1.919673,0.025388,-3.991663,0.219348,-4.179467,1
1,-0.591327,1.957000,0.178503,1.216105,-0.018648,-2.219300,0.085893,0.358234,-3.147598,-0.827189,1
2,-1.841737,2.239074,0.404437,0.105935,-2.589481,1.236131,0.965397,0.384895,-0.971101,-0.083446,0
3,-1.905812,NaN,-3.030398,-0.251969,0.420580,0.176821,-0.721738,-0.375486,-0.501078,-3.039635,0
4,1.124853,1.890323,0.701284,1.194078,1.814746,-0.334077,1.124353,0.176116,-2.036578,1.138263,1


## 2. Remove null values and standardize the data

This preprocessing step prepares the data for both baseline modeling and PCA.

What happens here:
- rows with null values are removed
- features and target are separated
- train and test sets are created
- standardization is applied using training data only

In [18]:
# Check missing values before cleaning.
print("Null values before cleaning:")
print(df.isnull().sum())

# Remove rows that contain null values.
clean_df = df.dropna().reset_index(drop=True)

# Split features and target.
X_clean = clean_df[feature_cols]
y_clean = clean_df["target"]

# Create train and test sets.
X_train, X_test, y_train, y_test = train_test_split(
    X_clean,
    y_clean,
    test_size=0.2,
    random_state=42,
    stratify=y_clean,
    )

# Standardize numeric features for modeling and PCA.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nDataset shape after cleaning:", clean_df.shape)
print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)

Null values before cleaning:
feature_1     0
feature_2     3
feature_3     0
feature_4     0
feature_5     0
feature_6     0
feature_7     2
feature_8     0
feature_9     0
feature_10    0
target        0
dtype: int64

Dataset shape after cleaning: (295, 11)
Training set shape: (236, 10)
Test set shape: (59, 10)


## 3. Train the baseline model

We first train a model on the standardized features without PCA.

Why this matters:
- it gives a reference accuracy
- later we can compare it with the PCA-based model

### Baseline model note

The next code cell trains logistic regression on the standardized full feature set.

This gives the reference accuracy before dimensionality reduction.

In [19]:
# Train logistic regression on the standardized full feature set.
baseline_model = LogisticRegression(max_iter=1000)
baseline_model.fit(X_train_scaled, y_train)

# Predict and print baseline accuracy.
baseline_predictions = baseline_model.predict(X_test_scaled)
baseline_accuracy = accuracy_score(y_test, baseline_predictions)

print(f"Baseline accuracy without PCA: {baseline_accuracy:.4f}")

Baseline accuracy without PCA: 0.8644


## 4. Apply PCA with 2 components

Now the same standardized data is reduced from 10 features to 2 principal components.

The next code cell fits PCA on training data and transforms both train and test sets.

In [25]:
# Fit PCA on the training data only.
pca = PCA(n_components=10)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print("Explained variance ratio for 2 components:", np.round(pca.explained_variance_ratio_, 4))
print("Total variance kept:", round(pca.explained_variance_ratio_.sum(), 4))

Explained variance ratio for 2 components: [0.2852 0.1574 0.1278 0.1111 0.1013 0.0858 0.0817 0.0496 0.     0.    ]
Total variance kept: 1.0


## 5. Train the model on PCA output

The next code cell trains the same logistic regression model using only the 2 PCA components.

That lets us compare baseline accuracy and PCA accuracy directly.

In [27]:
# Train logistic regression on the PCA-transformed data.
pca_model = LogisticRegression(max_iter=1000)
pca_model.fit(X_train_pca, y_train)

# Predict and print PCA-based accuracy.
pca_predictions = pca_model.predict(X_test_pca)
pca_accuracy = accuracy_score(y_test, pca_predictions)

print(f"Accuracy with PCA (n components): {pca_accuracy:.4f}")

Accuracy with PCA (n components): 0.8644


## 6. Final comparison

A small comparison table is added below so the two accuracies are easy to read together.

Focus on whether the PCA model stays close to the baseline model.

In [29]:
# Compare the two model accuracies side by side.
comparison_df = pd.DataFrame(
    {
        "model": ["Without PCA", f"With PCA ({pca.n_components} components)"],
        "accuracy": [baseline_accuracy, pca_accuracy],
    }
)

comparison_df

,model,accuracy
0,Without PCA,0.864407
1,With PCA (10 components),0.864407
